# WorldEnv × TRL GRPO — OpenEnv Hackathon

**Train an LLM agent on any world model projection using TRL's GRPOTrainer.**

WorldEnv wraps a [General Unified World Model](https://github.com/JacobFV/general-unified-world-modeling) (857 fields, 19 layers) as an OpenEnv environment factory. One model — infinite agent perspectives.

| Resource | Link |
|----------|------|
| 🌍 HF Space | [jacob-valdez/worldenv](https://huggingface.co/spaces/jacob-valdez/worldenv) |
| 🏢 Corporate Env | [jacob-valdez/corporate-world-env](https://huggingface.co/spaces/jacob-valdez/corporate-world-env) |
| 🤖 Robot Env | [jacob-valdez/robot-world-env](https://huggingface.co/spaces/jacob-valdez/robot-world-env) |
| 📖 Docs | [environments page](https://jacobfv.github.io/general-unified-world-modeling/environments/) |
| 💻 Source | [GitHub](https://github.com/JacobFV/general-unified-world-modeling) |

In [ ]:
# Install dependencies
!pip install -q openenv-core trl transformers torch accelerate
!pip install -q "general-unified-world-model[env]"  # optional: for local dynamics

In [ ]:
"""Step 1: Connect to WorldEnv and explore scenarios."""
import json, random
from openenv.core import EnvClient
from openenv.core.env_server.types import Action, Observation
from openenv.core.client_types import StepResult
from typing import Dict

WORLDENV_URL = "https://jacob-valdez-worldenv.hf.space"

# ── Minimal inline client (no extra install needed) ──────────────
from pydantic import Field
from openenv.core.env_server.types import Action, Observation

class WorldAction(Action):
    action_type: str = Field(default="step")
    target_field: str = Field(default="")
    value: float = Field(default=0.0)
    message: str = Field(default="")

class WorldObservation(Observation):
    visible_fields: Dict[str, float] = Field(default_factory=dict)
    step_count: int = Field(default=0)
    scenario_name: str = Field(default="")
    info: str = Field(default="")
    available_scenarios: list = Field(default_factory=list)
    obs_field_names: list = Field(default_factory=list)
    act_field_names: list = Field(default_factory=list)

class WorldEnv(EnvClient[WorldAction, WorldObservation]):
    def _step_payload(self, action):
        return {"action_type": action.action_type,
                "target_field": action.target_field,
                "value": action.value}
    def _parse_result(self, payload):
        obs_data = payload.get("observation", {})
        obs = WorldObservation(
            visible_fields=obs_data.get("visible_fields", {}),
            step_count=obs_data.get("step_count", 0),
            scenario_name=obs_data.get("scenario_name", ""),
            info=obs_data.get("info", ""),
            available_scenarios=obs_data.get("available_scenarios", []),
            obs_field_names=obs_data.get("obs_field_names", []),
            act_field_names=obs_data.get("act_field_names", []),
            done=payload.get("done", False),
            reward=payload.get("reward"),
        )
        from openenv.core.client_types import StepResult
        return StepResult(observation=obs, reward=payload.get("reward"), done=payload.get("done", False))
    def _parse_state(self, payload):
        from openenv.core.env_server.types import State
        return State(episode_id=payload.get("episode_id"), step_count=payload.get("step_count", 0))

# ── Connect and explore ──────────────────────────────────────────
with WorldEnv(base_url=WORLDENV_URL) as env:
    result = env.reset()
    obs = result.observation
    print(f"Scenario: {obs.scenario_name}")
    print(f"Available scenarios: {obs.available_scenarios}")
    print(f"Observable fields ({len(obs.obs_field_names)}): {obs.obs_field_names[:4]}...")
    print(f"Actionable fields: {obs.act_field_names}")
    print()
    print("Current field values:")
    for field, val in list(obs.visible_fields.items())[:6]:
        print(f"  {field}: {val:.4f}")
    print()

    # Take an action
    if obs.act_field_names:
        result = env.step(WorldAction(
            action_type="intervene",
            target_field=obs.act_field_names[0],
            value=0.8,
        ))
        print(f"After intervening: reward={result.reward:.4f}")

In [ ]:
"""Step 2: Define the reward function and rollout logic for GRPO."""
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import GRPOConfig, GRPOTrainer

SYSTEM_PROMPT = """You are an AI agent in a world simulation. You observe field values
and must take actions to maximize cumulative reward.

Available action_types: observe, predict, intervene, step
Respond ONLY with valid JSON:
{"action_type": "intervene", "target_field": "<field_path>", "value": <float>}
"""

def format_obs(obs) -> str:
    fields = "\n".join(f"  {k}: {v:.4f}" for k, v in list(obs.visible_fields.items())[:8])
    return (
        f"Scenario: {obs.scenario_name}\n"
        f"Step: {obs.step_count}\n"
        f"Fields:\n{fields}\n"
        f"You can act on: {obs.act_field_names}\n"
        f"{obs.info}"
    )

def parse_action(text: str, obs) -> WorldAction:
    text = text.strip().strip("```json").strip("```").strip()
    try:
        d = json.loads(text)
        return WorldAction(
            action_type=d.get("action_type", "step"),
            target_field=d.get("target_field", ""),
            value=float(d.get("value", 0.0)),
        )
    except Exception:
        if obs.act_field_names:
            return WorldAction(
                action_type="intervene",
                target_field=random.choice(obs.act_field_names),
                value=random.uniform(-1, 1),
            )
        return WorldAction(action_type="step")

print("Rollout logic defined.")

In [ ]:
"""Step 3: Load model and define GRPO rollout function."""

# Use a small model for this demo — swap for larger for real training
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print(f"Model loaded. Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")

# Persistent env client for training
env_client = WorldEnv(base_url=WORLDENV_URL)
env_client.connect()

N_STEPS_PER_EPISODE = 10  # steps per rollout episode

def rollout_func(prompts, trainer):
    """Custom rollout: run WorldEnv episodes and return (prompt, completion, reward) triples."""
    all_prompt_ids, all_completion_ids, all_rewards = [], [], []

    for _ in prompts:
        # Sample a fresh episode (random scenario)
        result = env_client.reset()
        obs = result.observation

        # Build prompt
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"World state:\n{format_obs(obs)}\n\nChoose an action:"},
        ]
        input_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(input_text, return_tensors="pt").to(trainer.model.device)

        # Generate
        with torch.no_grad():
            output = trainer.model.generate(
                **inputs, max_new_tokens=64, do_sample=True, temperature=0.8
            )
        completion_ids = output[0][inputs.input_ids.shape[1]:]
        completion_text = tokenizer.decode(completion_ids, skip_special_tokens=True)

        # Execute first action from LLM, then step N_STEPS_PER_EPISODE times
        action = parse_action(completion_text, obs)
        episode_reward = 0.0
        for _ in range(N_STEPS_PER_EPISODE):
            step_result = env_client.step(action)
            episode_reward += step_result.reward or 0.0
            if step_result.observation.done:
                break
            # Follow-up steps: random intervention
            if obs.act_field_names:
                action = WorldAction(
                    action_type="intervene",
                    target_field=random.choice(obs.act_field_names),
                    value=random.uniform(-1, 1),
                )
            else:
                action = WorldAction(action_type="step")

        all_prompt_ids.append(inputs.input_ids[0])
        all_completion_ids.append(completion_ids)
        all_rewards.append(episode_reward)

    return {
        "prompt_ids": all_prompt_ids,
        "completion_ids": all_completion_ids,
        "rewards": all_rewards,
    }

print("Rollout function defined.")

In [ ]:
"""Step 4: Configure and run GRPO training."""

training_args = GRPOConfig(
    output_dir="worldenv-agent",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    logging_steps=1,
    save_steps=50,
    max_completion_length=64,
    num_generations=4,
    bf16=True,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    tokenizer=tokenizer,
    rollout_func=rollout_func,
)

print("Starting WorldEnv GRPO training...")
print(f"Space: {WORLDENV_URL}")
trainer.train()

env_client.disconnect()
print("Done!")

In [ ]:
"""Step 5: Plot reward curve."""
import matplotlib.pyplot as plt

rewards = [log["reward"] for log in trainer.state.log_history if "reward" in log]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(rewards, color='#3498db', linewidth=1.5)
ax.fill_between(range(len(rewards)), rewards, alpha=0.2, color='#3498db')
ax.set_xlabel("Step")
ax.set_ylabel("Episode Reward")
ax.set_title("WorldEnv GRPO Training — Reward Curve")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("worldenv_reward_curve.png", dpi=150)
plt.show()
print(f"Final reward: {rewards[-1]:.4f}" if rewards else "No rewards logged yet")

## About WorldEnv

WorldEnv wraps the **General Unified World Model** — an 857-field typed causal ontology — as an OpenEnv environment factory. The `project()` function selects any field subset and compiles it onto a structured canvas with automatic attention topology. Each projection is a fully-formed RL environment.

**One model, infinite environments.**

```python
from general_unified_world_model import GeneralUnifiedWorldModel

model = GeneralUnifiedWorldModel(include=["financial", "regime", "sector_tech"])

# CEO environment
ceo_env = model.to_openenv(
    obs_fields=["firm.financials.revenue_growth", "firm.market.equity_price"],
    act_fields=["firm.strategy.capital_allocation"],
    reward_fn=lambda obs, act, info: obs["firm.financials.revenue_growth"].mean(),
)

# Employee environment — same dynamics, different perspective
employee_env = model.to_openenv(
    obs_fields=["person.state.stress", "person.state.confidence"],
    act_fields=["person.state.current_focus"],
    reward_fn=lambda obs, act, info: obs["person.state.confidence"].mean(),
)
```

- 🌍 [HF Space](https://huggingface.co/spaces/jacob-valdez/worldenv)
- 💻 [GitHub](https://github.com/JacobFV/general-unified-world-modeling)
- 📖 [Docs](https://jacobfv.github.io/general-unified-world-modeling/environments/)
- 📦 [PyPI](https://pypi.org/project/general-unified-world-model/)